# FlowNP en glucosa — v2: evaluación y resultados (Colab T4)

Entrenamiento (3 semillas) y E4 ya hechos con el notebook v1. Este notebook:
1. Prepara la sesión.
2. Muestra qué está hecho.
3. Evalúa las semillas elegidas (11 experimentos cada una).
4. Resume resultados frente a los baselines.
5. Genera tablas y figuras.

**En cada sesión nueva:** Entorno de ejecución → T4 GPU, y ejecutar las celdas 1 y 2. Todo es reanudable: lo terminado está en Drive y se salta.

**Si lo ejecuta un compañero desde su cuenta:** compartir la carpeta `PROYECTO` con permiso de edición, añadir un acceso directo en su "Mi unidad", ajustar `PROJ` y evaluar una semilla distinta (`SEEDS`).

## 1. Preparar la sesión

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/PROYECTO'   # <- ajusta si la carpeta está en otro sitio
%cd {PROJ}/flowNP
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
!pip -q install flow-matching==1.0.10 attrdictionary

## 2. Estado: entrenamiento, E4 y evaluación

In [ ]:
import os, json, glob
TASKS = ['e1_test', 'e2_nc5', 'e2_nc10', 'e2_nc20', 'e2_nc50',
         'e3_gap30', 'e3_gap60', 'e3_gap120', 'e3_gap180', 'e3_fc30', 'e3_fc60']
CK = 'output/results/glucose/default/fnp'
RES = '../glucose_np/results/FlowNP'
for seed in [0, 1, 2]:
    best = os.path.exists(f'{CK}/seed{seed}/best.tar')
    done = [t for t in TASKS if os.path.exists(f'{RES}/seed{seed}/{t}.json')]
    missing = [t for t in TASKS if t not in done]
    print(f'seed{seed}: modelo {"OK" if best else "FALTA"} | evaluados {len(done)}/11'
          + (f' | faltan: {", ".join(missing)}' if missing else ''))
print('E4:', 'OK' if os.path.exists(f'{RES}/e4_timing.json') else 'FALTA')

## 3. Evaluación
Configuración final: muestras con midpoint 20 pasos (elegido en E4); LL con midpoint 100 pasos + Hutchinson (error < 0,01 nats frente a la exacta con 400 pasos). ~1–1,5 h por semilla.

Cambia `SEEDS` para repartir el trabajo (p. ej. tú `[0, 1]` y un compañero `[2]`).

In [ ]:
SEEDS = [0, 1, 2]
for seed in SEEDS:
    !python glucose.py --mode=eval --seed={seed} --sample_method=midpoint --sample_steps=20 --num_samples=100 --ll_steps=100 --ll_div=hutchinson --eval_batch_size=64

## 4. Resumen de resultados (media ± desviación entre semillas evaluadas)

In [ ]:
import json, glob, numpy as np
KEYS = [('loglik_mgdl', 'LL'), ('rmse_mgdl', 'RMSE'), ('mae_mgdl', 'MAE'), ('crps_mgdl', 'CRPS'), ('cov90', 'cov90')]
MODELS = ['FlowNP', 'Interp. lineal', 'Media contexto']
print(f"{'experimento':10s} {'modelo':15s} {'n':>2s} " + ' '.join(f'{n:>14s}' for _, n in KEYS))
for t in TASKS:
    for m in MODELS:
        fs = sorted(glob.glob(f'../glucose_np/results/{m}/seed*/{t}.json'))
        if not fs:
            continue
        o = [json.load(open(f))['overall'] for f in fs]
        cols = [f'{np.mean([x[k] for x in o]):7.3f}±{np.std([x[k] for x in o]):5.3f}' for k, _ in KEYS]
        print(f'{t:10s} {m:15s} {len(fs):2d} ' + ' '.join(f'{c:>14s}' for c in cols))
    print()

## 5. Tablas y figuras (código de A)
Salen en `glucose_np/tables/` y `glucose_np/figures/`.

In [ ]:
%cd {PROJ}/glucose_np
!python plots.py
%cd {PROJ}/flowNP

## 6. Antes de cerrar
Para la celda que esté corriendo (■) y ejecuta esto para que Drive termine de guardar todo.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()